# 04 · Evaluation, threshold selection and explainability

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = ""
USE_DRIVE = True

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "scikit-learn", "scipy", "pandas", "matplotlib", "pillow", "tqdm"], check=True)
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.environ.setdefault("XRAY_ARTIFACTS", "/content/drive/MyDrive/xray_project/artifacts")
        os.environ.setdefault("XRAY_RESULTS", "/content/drive/MyDrive/xray_project/results")


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "config.py").exists():
            return p


ROOT = find_root()
if ROOT is None:
    assert REPO_URL, "Set REPO_URL or open the notebook from inside the repository"
    subprocess.run(["git", "clone", REPO_URL, "xray-vision-pneumonia-CNN"], check=True)
    ROOT = Path("xray-vision-pneumonia-CNN").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src import config as C
from src import dataset as D
from src import evaluate as E
from src import gradcam as G
from src import models as M
from src.utils import get_run, load_json, load_runs, save_json, sync_results_to_repo

C.ensure_dirs()
runs = load_runs()


def vp(rid, split):
    d = E.load_preds(rid, split)
    return d["y"], d["p"]

## Validation metrics for every run

In [ ]:
val_tbl = E.metrics_table(runs["run_id"].tolist(), "val").merge(
    runs[["run_id", "family", "variant", "strategy", "seed", "tag", "parent"]], on="run_id"
)
val_tbl.sort_values("roc_auc", ascending=False)[
    ["run_id", "accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc", "pr_auc", "spec_at_target_sens"]
].round(4).head(25)

## Model selection (validation only)

In [ ]:
FINAL_RUN_ID = None

cand = val_tbl[val_tbl["variant"].isin(["scratch", "finetune"])]
ranked = cand.sort_values(["roc_auc", "spec_at_target_sens"], ascending=False)
final_id = FINAL_RUN_ID or ranked.iloc[0]["run_id"]
runner_up = ranked[ranked["run_id"] != final_id].iloc[0]["run_id"]
best_scratch = cand[cand["family"] == "scratch"].sort_values("roc_auc", ascending=False).iloc[0]["run_id"]
best_ft = {a: g.sort_values("roc_auc", ascending=False).iloc[0]["run_id"] for a, g in cand[cand["family"] != "scratch"].groupby("family")}
by_id = runs.set_index("run_id")
heads = [by_id.loc[r, "parent"] for r in best_ft.values()]
REPORT_RUNS = list(dict.fromkeys([final_id, runner_up, best_scratch, *best_ft.values(), *heads]))
print("final:", final_id)
print("runner-up:", runner_up)
print("reported runs:", REPORT_RUNS)

In [ ]:
curves = {r: vp(r, "val") for r in REPORT_RUNS}
E.plot_roc(curves, C.FIG_DIR / "roc_val.png", title="ROC (validation)")
E.plot_pr(curves, C.FIG_DIR / "pr_val.png", title="Precision-recall (validation)")

## Operating threshold (validation only)

In [ ]:
yv, pv = vp(final_id, "val")
tt = E.threshold_table(yv, pv)
tt.to_csv(C.RESULTS_DIR / "threshold_table_val.csv", index=False)
thresholds = {r: E.choose_threshold(*vp(r, "val"), C.TARGET_SENSITIVITY)[0] for r in REPORT_RUNS}
save_json(
    {
        "final_run_id": final_id,
        "policy": f"highest validation threshold with sensitivity >= {C.TARGET_SENSITIVITY}",
        "target_sensitivity": C.TARGET_SENSITIVITY,
        "thresholds": thresholds,
        "validation_metrics": E.compute_metrics(yv, pv, thresholds[final_id]),
    },
    C.THRESHOLD_JSON,
)
tt.round(4)

## Test set (frozen): no model, threshold or hyper-parameter changes below this point

In [ ]:
for r in REPORT_RUNS:
    E.predict_run(r, "test")
test_thr = E.metrics_table(REPORT_RUNS, "test", thresholds=thresholds, include_oracle=False).assign(setting="validation-chosen threshold")
test_05 = E.metrics_table(REPORT_RUNS, "test", include_oracle=False).assign(setting="0.5")
val_rows = val_tbl.assign(setting="0.5")
pd.concat([val_rows, test_thr, test_05], ignore_index=True).to_csv(C.METRICS_CSV, index=False)
cols = ["run_id", "setting", "threshold", "tn", "fp", "fn", "tp", "accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc", "pr_auc"]
pd.concat([test_thr, test_05])[cols].round(4)

In [ ]:
rows = []
for r in REPORT_RUNS:
    y, p = vp(r, "test")
    ci = E.bootstrap_ci(y, p, thresholds[r])
    m = E.compute_metrics(y, p, thresholds[r])
    for k, (lo, hi) in ci.items():
        rows.append({"run_id": r, "metric": k, "value": m[k], "ci_low": lo, "ci_high": hi})
ci_tbl = pd.DataFrame(rows)
ci_tbl.to_csv(C.RESULTS_DIR / "test_bootstrap_ci.csv", index=False)
ci_tbl[ci_tbl["run_id"] == final_id].round(4)

In [ ]:
for r in REPORT_RUNS:
    y, p = vp(r, "test")
    E.plot_confusion(y, p, thresholds[r], C.FIG_DIR / f"cm_test_{r}.png", title=r)
curves_t = {r: vp(r, "test") for r in REPORT_RUNS}
E.plot_roc(curves_t, C.FIG_DIR / "roc_test.png", title="ROC (test)")
E.plot_pr(curves_t, C.FIG_DIR / "pr_test.png", title="Precision-recall (test)")

## Clinical cost of errors

In [ ]:
yt, pt = vp(final_id, "test")
mt = E.compute_metrics(yt, pt, thresholds[final_id])
print(f"missed pneumonia (FN): {mt['fn']} of {mt['tp'] + mt['fn']}   false alarms (FP): {mt['fp']} of {mt['tn'] + mt['fp']}")
screen = E.screening_table(mt["sensitivity"], mt["specificity"])
screen.to_csv(C.RESULTS_DIR / "screening_table.csv", index=False)
screen.round(3)

In [ ]:
yr, pr_ = vp(runner_up, "test")
mc = E.mcnemar_exact(yt, pt >= thresholds[final_id], pr_ >= thresholds[runner_up])
ad = E.paired_auc_diff(yt, pt, pr_)
sig = {"final": final_id, "runner_up": runner_up, "mcnemar": mc, "auc_diff": ad}
save_json(sig, C.RESULTS_DIR / "significance_final_vs_runner_up.json")
sig

## Calibration, test-time augmentation, selective prediction, ensemble

In [ ]:
T_fit = E.fit_temperature(*vp(final_id, "val"))
yv, pv = vp(final_id, "val")
cal = pd.DataFrame(
    {
        "split": ["val", "val", "test", "test"],
        "variant": ["raw", "temperature-scaled", "raw", "temperature-scaled"],
        "ece": [E.ece(yv, pv), E.ece(yv, E.apply_temperature(pv, T_fit)), E.ece(yt, pt), E.ece(yt, E.apply_temperature(pt, T_fit))],
        "brier": [E.compute_metrics(y_, p_)["brier"] for y_, p_ in ((yv, pv), (yv, E.apply_temperature(pv, T_fit)), (yt, pt), (yt, E.apply_temperature(pt, T_fit)))],
        "log_loss": [E.compute_metrics(y_, p_)["log_loss"] for y_, p_ in ((yv, pv), (yv, E.apply_temperature(pv, T_fit)), (yt, pt), (yt, E.apply_temperature(pt, T_fit)))],
    }
)
cal["temperature"] = T_fit
cal.to_csv(C.RESULTS_DIR / "calibration.csv", index=False)
E.plot_reliability({"val raw": (yv, pv), "test raw": (yt, pt), "test scaled": (yt, E.apply_temperature(pt, T_fit))}, C.FIG_DIR / "reliability.png")
cal.round(4)

In [ ]:
tta_rows = []
for split in ("val", "test"):
    d0 = E.load_preds(final_id, split)
    d1 = E.predict_run(final_id, split, tta=True)
    thr_ = thresholds[final_id] if split == "val" else None
    tta_rows.append({"split": split, "tta": False, **{k: E.compute_metrics(d0["y"], d0["p"], thresholds[final_id])[k] for k in ("roc_auc", "pr_auc", "log_loss")}})
    tta_rows.append({"split": split, "tta": True, **{k: E.compute_metrics(d1["y"], d1["p"], thresholds[final_id])[k] for k in ("roc_auc", "pr_auc", "log_loss")}})
tta_tbl = pd.DataFrame(tta_rows)
tta_tbl.to_csv(C.RESULTS_DIR / "tta.csv", index=False)
tta_tbl.round(4)

In [ ]:
sel = E.selective_curve(yt, pt, thresholds[final_id])
sel.to_csv(C.RESULTS_DIR / "selective_prediction_test.csv", index=False)
E.plot_selective(sel, C.FIG_DIR / "selective_prediction.png")
sel.round(3)

In [ ]:
members = list(best_ft.values())
ens_val = E.ensemble_probs([vp(r, "val")[1] for r in members])
ens_test = E.ensemble_probs([vp(r, "test")[1] for r in members])
ens_thr = E.choose_threshold(yv, ens_val, C.TARGET_SENSITIVITY)[0]
ens = pd.DataFrame(
    [
        {"split": "val", **E.compute_metrics(yv, ens_val, ens_thr)},
        {"split": "test", **E.compute_metrics(yt, ens_test, ens_thr)},
    ]
)
ens.insert(0, "members", "+".join(members))
ens.to_csv(C.RESULTS_DIR / "ensemble.csv", index=False)
ens[["split", "threshold", "fn", "fp", "sensitivity", "specificity", "roc_auc", "pr_auc"]].round(4)

## Error analysis

In [ ]:
test_df = D.get_splits()["test"].copy()
thr = thresholds[final_id]
test_df["p"] = pt
test_df["pred"] = (pt >= thr).astype(int)
stats = pd.read_csv(C.IMAGE_STATS_CSV)
test_df = test_df.merge(stats[["rel_path", "mean", "std", "marker_score", "file_bytes", "aspect"]], on="rel_path", how="left")
test_df["error"] = test_df["pred"] != test_df["label"]
test_df["error_type"] = np.select(
    [(test_df["label"] == 1) & (test_df["pred"] == 0), (test_df["label"] == 0) & (test_df["pred"] == 1)], ["FN", "FP"], "correct"
)
by_sub = test_df.groupby("subtype").agg(n=("error", "size"), errors=("error", "sum"))
by_sub["error_rate"] = by_sub["errors"] / by_sub["n"]
by_sub.to_csv(C.RESULTS_DIR / "errors_by_subtype.csv")
by_type = test_df.groupby("error_type")[["mean", "std", "width", "height", "aspect", "marker_score", "p"]].mean()
by_type.to_csv(C.RESULTS_DIR / "errors_by_type.csv")
test_df[test_df["error"]][["rel_path", "label", "subtype", "p", "error_type"]].to_csv(C.RESULTS_DIR / "errors_list.csv", index=False)
display(by_sub.round(3))
display(by_type.round(3))

In [ ]:
root = D.find_data_root()
fn = test_df[test_df["error_type"] == "FN"].sort_values("p")
fp = test_df[test_df["error_type"] == "FP"].sort_values("p", ascending=False).head(12)
E.plot_gallery(fn["rel_path"].tolist()[:16], [f"FN {s} p={p:.2f}" for s, p in zip(fn["subtype"], fn["p"])][:16], root, C.FIG_DIR / "errors_fn.png", suptitle="False negatives (lowest P(pneumonia) first)")
E.plot_gallery(fp["rel_path"].tolist(), [f"FP p={p:.2f}" for p in fp["p"]], root, C.FIG_DIR / "errors_fp.png", suptitle="False positives (highest P(pneumonia) first)")

## Grad-CAM and artefact checks

In [ ]:
final_row = get_run(final_id)
size = tuple(json.loads(final_row["image_size"]))
resize = final_row["resize"]
final_model = M.load_run_model(final_row)
scratch_model = M.load_run_model(get_run(best_scratch))

ex = E.select_examples(yt, pt, thr)
test_df = test_df.reset_index(drop=True)
marker_idx = test_df.sort_values("marker_score", ascending=False).index[:4].tolist()
groups = {**ex, "MARKER": marker_idx}


def make_items(idx, tag, probs, threshold):
    out = []
    for i in idx:
        pred = "pneumonia" if probs[i] >= threshold else "normal"
        out.append({"path": test_df["path"][i], "target": pred, "title": f"{tag} | true {C.CLASS_NAMES[test_df['label'][i]]} | P={probs[i]:.2f} | cam: {pred}"})
    return out


for name, idx in groups.items():
    if idx:
        G.cam_figure(final_model, make_items(idx, name, pt, thr), C.FIG_DIR / f"gradcam_final_{name}.png", methods=("gradcam", "gradcampp", "occlusion"), resize=resize, image_size=size)

In [ ]:
ys, ps = vp(best_scratch, "test")
thr_s = thresholds[best_scratch]
for name in ("FN", "FP"):
    idx = E.select_examples(ys, ps, thr_s)[name][:4]
    if idx:
        G.cam_figure(scratch_model, make_items(idx, f"scratch {name}", ps, thr_s), C.FIG_DIR / f"gradcam_scratch_{name}.png", resize=get_run(best_scratch)["resize"], image_size=tuple(json.loads(get_run(best_scratch)["image_size"])))

In [ ]:
sample = test_df.sample(min(120, len(test_df)), random_state=C.SEED)
rows = []
for i in sample.index:
    img = D.load_image(test_df["path"][i], size, resize)
    target = "pneumonia" if pt[i] >= thr else "normal"
    cam = G.gradcam(final_model, img, target)
    rows.append({"label": C.CLASS_NAMES[test_df["label"][i]], "error_type": test_df["error_type"][i], **G.cam_mass_fractions(cam)})
quant = pd.DataFrame(rows)
quant_tbl = quant.groupby(["label", "error_type"]).mean(numeric_only=True)
quant_tbl.to_csv(C.RESULTS_DIR / "gradcam_region_mass.csv")
quant_tbl.round(3)

In [ ]:
splits = D.get_splits()
val_df = splits["val"]
yv_, pv_ = vp(final_id, "val")
probes = [{"probe": "none", "roc_auc": roc_auc_score(yv_, pv_)}]
for region, mode in (("center", "remove"), ("center", "keep"), ("top_corners", "remove"), ("border", "remove")):
    p = G.predict_masked(final_model, val_df, region, mode, resize, size)
    probes.append({"probe": f"{region}-{mode}", "roc_auc": roc_auc_score(val_df["label"], p)})
probes = pd.DataFrame(probes)
probes["delta_vs_none"] = probes["roc_auc"] - probes.loc[0, "roc_auc"]
probes.to_csv(C.RESULTS_DIR / "artefact_probes_val.csv", index=False)
probes.round(4)

In [ ]:
random_model = M.build_from_cfg(json.loads(final_row["model_cfg"]), pretrained=False)
imgs = [D.load_image(val_df["path"][i], size, resize) for i in range(0, len(val_df), max(1, len(val_df) // 10))][:10]
rho = G.randomization_check(final_model, random_model, imgs)
pd.DataFrame({"spearman_trained_vs_random": rho}).describe().to_csv(C.RESULTS_DIR / "gradcam_randomization_check.csv")
print("mean Spearman between trained and randomised-model maps:", np.nanmean(rho))

In [ ]:
print(load_json(C.TEST_LOG_JSON))
sync_results_to_repo()